# Tracker ReID evaluation on MOT17 val

Compare BoT-SORT with and without a ReID encoder on the MOT17 val-half split (YOLOX detections, TrackEval metrics).

| Config | Tracker | CMC | ReID | Fusion |
|---|---|---|---|---|
| Baseline | BoT-SORT | ✓ (sparseOptFlow) | ✗ | geometry + CMC |
| + ReID | BoT-SORT | ✓ | ✓ | min-cost (§5) |

**Defaults:** `fastreid_mot17_sbs50`, `REID_APPEARANCE_THRESHOLD=0.2` (MOT17 re-ID study Table 8).

**Data:** `trackers download mot17 --split val --asset annotations,frames` + [YOLOX val detections](https://drive.google.com/file/d/1BuXtPWf8QbPU_y1i2xY2IbTE-rj3l6qT).

**Outputs:** `trackers_reid_outputs/` — set `RERUN[name]=False` to reuse cached preds.

§8 builds a side-by-side baseline vs +ReID video for the sequence with the largest ΔHOTA (Colab auto-download).

> **Runtime:** T4 GPU.


## 1. Setup

Install `trackers` from the feature branch. On Colab, use the cells below instead of `trackers[reid]` — Colab already ships CUDA PyTorch and the extra pulls a conflicting build.


In [ ]:
GIT_REF = "git+https://github.com/roboflow/trackers.git@release/stable"

!pip install -q --upgrade pip
# Colab ships CUDA torch — avoid trackers[reid] (pulls a conflicting PyPI torch build).
!pip install -q timm huggingface-hub safetensors gdown matplotlib scikit-learn
!pip install -q --no-cache-dir --force-reinstall --no-deps "trackers @ {GIT_REF}"
!pip install -q supervision scipy opencv-python rich requests pydeprecate

In [ ]:
from __future__ import annotations

import subprocess
import warnings
import zipfile
from pathlib import Path

import cv2
import gdown
import matplotlib.pyplot as plt
import numpy as np
import supervision as sv
import torch
from IPython.display import Video, display
from sklearn.decomposition import PCA

from trackers import BoTSORTTracker
from trackers.core.reid import ReIDModel
from trackers.core.reid.models.registry import DEFAULT_MODEL, FASTREID_MOT17_SBS50
from trackers.eval import evaluate_mot_sequences
from trackers.eval.box import box_iou
from trackers.eval.results import BenchmarkResult
from trackers.io.mot import _MOTOutput, load_mot_file

warnings.filterwarnings("ignore")

try:
    from google.colab import files

    IN_COLAB = True
    REPO_ROOT = Path("/content")
except ImportError:
    files = None
    IN_COLAB = False
    REPO_ROOT = Path("..").resolve()

VAL_SEQUENCES = [
    "MOT17-02-FRCNN",
    "MOT17-04-FRCNN",
    "MOT17-05-FRCNN",
    "MOT17-09-FRCNN",
    "MOT17-10-FRCNN",
    "MOT17-11-FRCNN",
    "MOT17-13-FRCNN",
]

device = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()} | {device}")

## 2. ReID model

| `REID_ENCODER` | Training | Input |
|---|---|---|
| `fastreid_mot17_sbs50` (default) | MOT17 train-half | 384×128 |
| `osnet_msmt17` | MSMT17 combineall | 256×128 |


In [ ]:
REID_ENCODER = "fastreid_mot17_sbs50"
REID_APPEARANCE_THRESHOLD = 0.2  # MOT17 re-ID study Table 8; BoT-SORT paper default 0.25

if REID_ENCODER == FASTREID_MOT17_SBS50:
    reid_model = ReIDModel.from_pretrained(FASTREID_MOT17_SBS50)
elif REID_ENCODER in (DEFAULT_MODEL, "osnet_msmt17"):
    reid_model = ReIDModel.from_pretrained()
else:
    raise ValueError(f"Unknown REID_ENCODER: {REID_ENCODER!r}")

print(f"Encoder: {REID_ENCODER}  |  θ_emb: {REID_APPEARANCE_THRESHOLD}")
print(reid_model.preprocessing.describe())

## 3. Download data

MOT17 val GT + frames via `trackers download`. YOLOX val detections via gdown
(BoT-SORT / ByteTrack eval protocol). YOLOX frame IDs are remapped to 1…N.


In [ ]:
FORCE_DOWNLOAD = False

MOT17_VAL = REPO_ROOT / "mot17" / "val"
YOLOX_DIR = REPO_ROOT / "MOT17_yolox_dets"
YOLOX_VAL_DIR = YOLOX_DIR / "val"
YOLOX_ZIP = YOLOX_DIR / "yolox_detections_MOT17.zip"
YOLOX_GDRIVE_ID = "1BuXtPWf8QbPU_y1i2xY2IbTE-rj3l6qT"
OUTPUT_ROOT = REPO_ROOT / "trackers_reid_outputs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


def yolox_det_path(seq: str) -> Path:
    return YOLOX_VAL_DIR / f"{seq.replace('-FRCNN', '')}_val.txt"


def mot17_val_ready() -> bool:
    return all(
        (MOT17_VAL / seq / "gt" / "gt.txt").is_file() and (MOT17_VAL / seq / "img1").is_dir() for seq in VAL_SEQUENCES
    )


def yolox_ready() -> bool:
    return YOLOX_VAL_DIR.is_dir() and len(list(YOLOX_VAL_DIR.glob("MOT17-*_val.txt"))) >= len(VAL_SEQUENCES)


if FORCE_DOWNLOAD or not mot17_val_ready():
    subprocess.run(
        [
            "trackers",
            "download",
            "mot17",
            "--split",
            "val",
            "--asset",
            "annotations,frames",
            "-o",
            str(REPO_ROOT),
        ],
        check=True,
    )
else:
    print("MOT17 val already present.")

if FORCE_DOWNLOAD or not yolox_ready():
    YOLOX_DIR.mkdir(parents=True, exist_ok=True)
    print("Downloading YOLOX val detections…")
    gdown.download(id=YOLOX_GDRIVE_ID, output=str(YOLOX_ZIP), quiet=False)
    with zipfile.ZipFile(YOLOX_ZIP) as zf:
        zf.extractall(YOLOX_DIR)
else:
    print("YOLOX detections already present.")

SEQUENCE_PATHS: dict[str, dict] = {}
for seq in VAL_SEQUENCES:
    gt = MOT17_VAL / seq / "gt" / "gt.txt"
    img = MOT17_VAL / seq / "img1"
    det = yolox_det_path(seq)
    if not (gt.is_file() and img.is_dir() and det.is_file()):
        print(f"  skip {seq}: missing gt, img1, or YOLOX det")
        continue
    n_frames = len(list(img.glob("*.jpg")))
    SEQUENCE_PATHS[seq] = {"gt": gt, "img": img, "det": det, "n_frames": n_frames}
    print(f"  {seq}: {n_frames} frames")

ACTIVE_SEQUENCES = list(SEQUENCE_PATHS)
if not ACTIVE_SEQUENCES:
    raise RuntimeError("No sequences ready — re-run downloads above.")

SEQMAP_PATH = OUTPUT_ROOT / "MOT17-val.txt"
SEQMAP_PATH.write_text("name\n" + "\n".join(ACTIVE_SEQUENCES) + "\n")
print(f"\n{len(ACTIVE_SEQUENCES)} sequences → outputs in {OUTPUT_ROOT}")

## 4. Tracking helpers


In [ ]:
RERUN = {
    "botsort_baseline": True,
    "botsort_reid": True,
}


def _yolox_frame_offset(det_path: Path) -> int:
    min_frame = None
    with det_path.open() as f:
        for line in f:
            parts = line.strip().split(",")
            if len(parts) < 6:
                continue
            frame = int(float(parts[0]))
            min_frame = frame if min_frame is None else min(min_frame, frame)
    return (min_frame - 1) if min_frame and min_frame > 1 else 0


def load_yolox_dets(det_path: Path) -> dict[int, sv.Detections]:
    offset = _yolox_frame_offset(det_path)
    by_frame: dict[int, list[list[float]]] = {}
    with det_path.open() as f:
        for line in f:
            parts = line.strip().split(",")
            if len(parts) < 6:
                continue
            frame = int(float(parts[0])) - offset
            if frame < 1:
                continue
            x1, y1, x2, y2, score = map(float, parts[1:6])
            if score <= 0:
                continue
            by_frame.setdefault(frame, []).append([x1, y1, x2, y2, score])
    return {
        frame: sv.Detections(
            xyxy=np.array(boxes, dtype=np.float32)[:, :4],
            confidence=np.array(boxes, dtype=np.float32)[:, 4],
        )
        for frame, boxes in by_frame.items()
    }


def fmt_metrics(result: BenchmarkResult) -> tuple[float, float, float, float, float, int]:
    a = result.aggregate
    return (
        (a.HOTA.HOTA * 100 if a.HOTA else float("nan")),
        (a.HOTA.AssA * 100 if a.HOTA else float("nan")),
        (a.HOTA.DetA * 100 if a.HOTA else float("nan")),
        (a.CLEAR.MOTA * 100 if a.CLEAR else float("nan")),
        (a.Identity.IDF1 * 100 if a.Identity else float("nan")),
        (a.CLEAR.IDSW if a.CLEAR else 0),
    )


def print_metrics(label: str, result: BenchmarkResult) -> None:
    hota, assa, deta, mota, idf1, idsw = fmt_metrics(result)
    print(f"{label}: HOTA {hota:6.2f}  MOTA {mota:6.2f}  IDF1 {idf1:6.2f}  IDSW {idsw}")


def run_tracking(name: str, factory, *, use_frames: bool) -> Path:
    pred_dir = OUTPUT_ROOT / name / "preds"
    pred_dir.mkdir(parents=True, exist_ok=True)

    for seq in ACTIVE_SEQUENCES:
        spec = SEQUENCE_PATHS[seq]
        dets = load_yolox_dets(spec["det"])
        images = sorted(spec["img"].glob("*.jpg"))
        tracker = factory()

        with _MOTOutput(pred_dir / f"{seq}.txt") as out:
            for frame_idx in range(1, spec["n_frames"] + 1):
                frame = None
                if use_frames and frame_idx <= len(images):
                    frame = cv2.imread(str(images[frame_idx - 1]))
                tracked = tracker.update(dets.get(frame_idx, sv.Detections.empty()), frame)
                if tracked.tracker_id is not None:
                    tracked = tracked[tracked.tracker_id != -1]
                out.write(frame_idx, tracked)
        print(f"  {seq}: {spec['n_frames']} frames")

    return pred_dir


def evaluate(name: str, pred_dir: Path) -> BenchmarkResult:
    result = evaluate_mot_sequences(
        gt_dir=MOT17_VAL,
        tracker_dir=pred_dir,
        seqmap=SEQMAP_PATH,
        metrics=["CLEAR", "HOTA", "Identity"],
    )
    cache = OUTPUT_ROOT / name / "eval_results.json"
    cache.parent.mkdir(parents=True, exist_ok=True)
    result.save(cache)
    return result


def load_or_run(name: str, factory, *, use_frames: bool) -> BenchmarkResult:
    pred_dir = OUTPUT_ROOT / name / "preds"
    cache = OUTPUT_ROOT / name / "eval_results.json"
    preds_ok = pred_dir.exists() and all((pred_dir / f"{s}.txt").exists() for s in ACTIVE_SEQUENCES)

    ran = False
    if RERUN.get(name, True) or not preds_ok:
        print(f"Running {name}…")
        pred_dir = run_tracking(name, factory, use_frames=use_frames)
        ran = True
    else:
        print(f"Using cached preds: {pred_dir}")

    if not ran and cache.exists():
        print(f"Using cached eval: {cache}")
        return BenchmarkResult.load(cache)

    print(f"Evaluating {name}…")
    return evaluate(name, pred_dir)


def match_dets_to_gt(gt_frame, det_xyxy: np.ndarray, min_iou: float = 0.5) -> np.ndarray:
    if len(det_xyxy) == 0:
        return np.array([], dtype=np.int64)
    gt_xyxy = sv.xywh_to_xyxy(gt_frame.boxes)
    keep = (gt_frame.confidences > 0) & (gt_frame.classes == 1)
    gt_xyxy, gt_ids = gt_xyxy[keep], gt_frame.ids[keep]
    if len(gt_xyxy) == 0:
        return np.full(len(det_xyxy), -1, dtype=np.int64)
    ious = box_iou(det_xyxy.astype(np.float64), gt_xyxy.astype(np.float64))
    out = np.full(len(det_xyxy), -1, dtype=np.int64)
    for i in range(len(det_xyxy)):
        j = int(np.argmax(ious[i]))
        if ious[i, j] >= min_iou:
            out[i] = int(gt_ids[j])
    return out

## 5. Run trackers


In [ ]:
EXPERIMENTS = [
    (
        "botsort_baseline",
        "BoT-SORT (baseline)",
        lambda: BoTSORTTracker(enable_cmc=True),
        True,
    ),
    (
        "botsort_reid",
        "BoT-SORT + ReID",
        lambda: BoTSORTTracker(
            enable_cmc=True,
            reid_model=reid_model,
            reid_ema_alpha=0.9,
            appearance_threshold=REID_APPEARANCE_THRESHOLD,
        ),
        True,
    ),
]

results: dict[str, BenchmarkResult] = {}
for name, label, factory, use_frames in EXPERIMENTS:
    results[name] = load_or_run(name, factory, use_frames=use_frames)
    print_metrics(label, results[name])
    print()

result_baseline = results["botsort_baseline"]
result_reid = results["botsort_reid"]

## 6. ReID embedding visualization (optional)


In [ ]:
VIZ_SEQ = "MOT17-02-FRCNN"
VIZ_STRIDE, VIZ_MAX_FRAMES, VIZ_MAX_POINTS, VIZ_MAX_CROPS = 5, 40, 300, 24

spec = SEQUENCE_PATHS[VIZ_SEQ]
gt_by_frame = load_mot_file(spec["gt"])
dets_by_frame = load_yolox_dets(spec["det"])
images = sorted(spec["img"].glob("*.jpg"))

crops, embeddings, gt_ids = [], [], []
for frame_idx in list(range(1, spec["n_frames"] + 1, VIZ_STRIDE))[:VIZ_MAX_FRAMES]:
    dets = dets_by_frame.get(frame_idx)
    gt = gt_by_frame.get(frame_idx)
    if dets is None or gt is None or len(dets) == 0:
        continue
    dets = dets[dets.confidence >= 0.5]
    if len(dets) == 0:
        continue
    bgr = cv2.imread(str(images[frame_idx - 1]))
    if bgr is None:
        continue
    matched = match_dets_to_gt(gt, dets.xyxy)
    feats = reid_model.extract_features(dets, bgr)
    for i in range(len(dets)):
        if matched[i] < 0:
            continue
        crop = sv.crop_image(bgr, dets.xyxy[i].astype(int))
        if crop.size == 0:
            continue
        crops.append(crop[:, :, ::-1])
        embeddings.append(feats[i])
        gt_ids.append(int(matched[i]))

if not embeddings:
    raise RuntimeError("No matched crops — try another sequence or lower confidence threshold")

emb = np.stack(embeddings)
labels = np.array(gt_ids)
if len(emb) > VIZ_MAX_POINTS:
    idx = np.linspace(0, len(emb) - 1, VIZ_MAX_POINTS, dtype=int)
    emb, labels, crops = emb[idx], labels[idx], [crops[i] for i in idx]

coords = PCA(n_components=2, random_state=0).fit_transform(emb)
unique = np.unique(labels)
colors = {pid: plt.colormaps["tab20"](i % 20) for i, pid in enumerate(unique)}

fig, (ax_pca, ax_crop) = plt.subplots(1, 2, figsize=(14, 6))
for pid in unique:
    m = labels == pid
    ax_pca.scatter(coords[m, 0], coords[m, 1], s=28, alpha=0.85, color=colors[pid], label=f"id {pid}")
ax_pca.set(title=f"{VIZ_SEQ} — PCA by GT id", xlabel="PC1", ylabel="PC2")
ax_pca.grid(True, alpha=0.3)
if len(unique) <= 12:
    ax_pca.legend(fontsize=8)

n_show = min(len(crops), VIZ_MAX_CROPS)
ncols, nrows = 6, int(np.ceil(n_show / 6))
mosaic = np.full((nrows * 64, ncols * 32, 3), 255, dtype=np.uint8)
for k in range(n_show):
    r, c = divmod(k, ncols)
    tile = cv2.resize(crops[k], (32, 64))
    y, x = r * 64, c * 32
    mosaic[y : y + 64, x : x + 32] = tile
    rgb = (np.array(colors[labels[k]])[:3] * 255).astype(np.uint8)
    mosaic[y : y + 2, x : x + 32] = rgb
    mosaic[y + 62 : y + 64, x : x + 32] = rgb

ax_crop.imshow(mosaic)
ax_crop.set(title=f"Sample crops ({n_show})")
ax_crop.axis("off")
plt.tight_layout()
plt.show()
print(f"{len(coords)} points, {len(unique)} GT ids")

## 7. Results

**7.1–7.2** BoT-SORT vs published references.


### 7.1 BoT-SORT — reference targets

**Primary — [*Does Re-ID Really Help in Multi-Object Tracking?*](https://www-sop.inria.fr/members/Francois.Bremond/Postscript/Tomasz__SCCAI_2025.pdf) (2025). BoT-SORT + YOLOX + MOT17 FastReID, app th=0.2. Combined val scores from **Table 8 (HOTA)** and **Table 13 (IDF1)**; MOTA is not reported for this YOLOX setup.

| Config | HOTA | IDF1 |
|---|---:|---:|
| No re-ID | 68.43 | 80.92 |
| MOT17 FastReID, app th=0.2 | 68.95 | 81.98 |
| **ReID Δ (reference)** | **+0.52** | **+1.06** |

**Secondary — [BoT-SORT paper](https://arxiv.org/abs/2206.14651)** (Table 1, MOT17 val):

| Method | HOTA | MOTA | IDF1 |
|---|---:|---:|---:|
| BoT-SORT | 69.11 | 78.39 | 81.53 |
| BoT-SORT + ReID | 69.17 | 78.46 | 82.07 |
| **ReID Δ (BoT-SORT paper)** | **+0.06** | **+0.07** | **+0.54** |


In [ ]:
# MOT17 re-ID study reference — Table 8 (HOTA) + Table 13 (IDF1), COMBINED row.
# MOTA is not reported for the YOLOX setup in that study.
REID_STUDY_NO_REID = {"hota": 68.428, "mota": None, "idf1": 80.92}
REID_STUDY_MOT17_TH02 = {"hota": 68.951, "mota": None, "idf1": 81.984}
REID_STUDY_REID_DELTA = {k: REID_STUDY_MOT17_TH02[k] - REID_STUDY_NO_REID[k] for k in ("hota", "idf1")}

# BoT-SORT paper Table 1 (MOT17 val, YOLOX).
BOTSORT_PAPER = {"hota": 69.11, "mota": 78.39, "idf1": 81.53}
BOTSORT_PAPER_REID = {"hota": 69.17, "mota": 78.46, "idf1": 82.07}
BOTSORT_PAPER_REID_DELTA = {k: BOTSORT_PAPER_REID[k] - BOTSORT_PAPER[k] for k in BOTSORT_PAPER}


def fmt_ref_metric(value: float | None) -> str:
    return f"{value:6.2f}" if value is not None else "     —"


def seq_metrics(result: BenchmarkResult, seq: str) -> tuple[float, float, float, int]:
    s = result.sequences.get(seq)
    if s is None:
        return float("nan"), float("nan"), float("nan"), 0
    return (
        s.HOTA.HOTA * 100 if s.HOTA else float("nan"),
        s.HOTA.AssA * 100 if s.HOTA else float("nan"),
        s.Identity.IDF1 * 100 if s.Identity else float("nan"),
        s.CLEAR.IDSW if s.CLEAR else 0,
    )


botsort_rows = [
    ("BoT-SORT (baseline)", result_baseline),
    ("BoT-SORT + ReID", result_reid),
]

print("BoT-SORT — trackers (aggregate, all val sequences)")
print(f"{'Config':<28}  {'HOTA':>6}  {'AssA':>6}  {'DetA':>6}  {'MOTA':>6}  {'IDF1':>6}  {'IDSW':>5}")
print("-" * 72)
for label, res in botsort_rows:
    hota, assa, deta, mota, idf1, idsw = fmt_metrics(res)
    print(f"{label:<28}  {hota:6.2f}  {assa:6.2f}  {deta:6.2f}  {mota:6.2f}  {idf1:6.2f}  {idsw:5d}")

b = fmt_metrics(result_baseline)
r = fmt_metrics(result_reid)
print(
    f"\nBoT-SORT ReID uplift (trackers): "
    f"ΔHOTA {r[0] - b[0]:+6.2f}  ΔMOTA {r[3] - b[3]:+6.2f}  "
    f"ΔIDF1 {r[4] - b[4]:+6.2f}  ΔIDSW {int(r[5] - b[5]):+5d}"
)

print("\nBoT-SORT vs MOT17 re-ID study (primary — Table 8 + Table 13)")
print(f"{'':28}  {'HOTA':>6}  {'MOTA':>6}  {'IDF1':>6}")
print("-" * 52)
print(
    f"{'Reference (no re-ID)':<28}  "
    f"{REID_STUDY_NO_REID['hota']:6.2f}  {fmt_ref_metric(REID_STUDY_NO_REID['mota'])}  "
    f"{REID_STUDY_NO_REID['idf1']:6.2f}"
)
print(
    f"{'trackers (baseline)':<28}  {b[0]:6.2f}  {b[3]:6.2f}  {b[4]:6.2f}  "
    f"  Δ {b[0] - REID_STUDY_NO_REID['hota']:+5.2f}  "
    f"{'—':>6}  {b[4] - REID_STUDY_NO_REID['idf1']:+5.2f}"
)
print(
    f"{'Reference (MOT17 th=0.2)':<28}  "
    f"{REID_STUDY_MOT17_TH02['hota']:6.2f}  {fmt_ref_metric(REID_STUDY_MOT17_TH02['mota'])}  "
    f"{REID_STUDY_MOT17_TH02['idf1']:6.2f}"
)
print(
    f"{'trackers (+ ReID)':<28}  {r[0]:6.2f}  {r[3]:6.2f}  {r[4]:6.2f}  "
    f"  Δ {r[0] - REID_STUDY_MOT17_TH02['hota']:+5.2f}  "
    f"{'—':>6}  {r[4] - REID_STUDY_MOT17_TH02['idf1']:+5.2f}"
)
print(
    f"\nReID uplift vs reference study\n"
    f"  ΔHOTA  trackers {r[0] - b[0]:+6.2f}   reference {REID_STUDY_REID_DELTA['hota']:+6.2f}   "
    f"gap {(r[0] - b[0]) - REID_STUDY_REID_DELTA['hota']:+6.2f}\n"
    f"  ΔMOTA  trackers {r[3] - b[3]:+6.2f}   reference      —\n"
    f"  ΔIDF1  trackers {r[4] - b[4]:+6.2f}   reference {REID_STUDY_REID_DELTA['idf1']:+6.2f}   "
    f"gap {(r[4] - b[4]) - REID_STUDY_REID_DELTA['idf1']:+6.2f}"
)

print("\nBoT-SORT vs BoT-SORT paper Table 1 (secondary)")
print(f"{'':28}  {'HOTA':>6}  {'MOTA':>6}  {'IDF1':>6}")
print("-" * 52)
print(
    f"{'BoT-SORT paper':<28}  {BOTSORT_PAPER['hota']:6.2f}  {BOTSORT_PAPER['mota']:6.2f}  {BOTSORT_PAPER['idf1']:6.2f}"
)
print(
    f"{'trackers (baseline)':<28}  {b[0]:6.2f}  {b[3]:6.2f}  {b[4]:6.2f}  "
    f"  Δ {b[0] - BOTSORT_PAPER['hota']:+5.2f}  "
    f"{b[3] - BOTSORT_PAPER['mota']:+5.2f}  {b[4] - BOTSORT_PAPER['idf1']:+5.2f}"
)
print(
    f"{'BoT-SORT paper + ReID':<28}  {BOTSORT_PAPER_REID['hota']:6.2f}  "
    f"{BOTSORT_PAPER_REID['mota']:6.2f}  {BOTSORT_PAPER_REID['idf1']:6.2f}"
)
print(
    f"{'trackers (+ ReID)':<28}  {r[0]:6.2f}  {r[3]:6.2f}  {r[4]:6.2f}  "
    f"  Δ {r[0] - BOTSORT_PAPER_REID['hota']:+5.2f}  "
    f"{r[3] - BOTSORT_PAPER_REID['mota']:+5.2f}  {r[4] - BOTSORT_PAPER_REID['idf1']:+5.2f}"
)
print(
    f"\nReID uplift vs BoT-SORT paper\n"
    f"  ΔHOTA  trackers {r[0] - b[0]:+6.2f}   BoT-SORT paper {BOTSORT_PAPER_REID_DELTA['hota']:+6.2f}   "
    f"gap {(r[0] - b[0]) - BOTSORT_PAPER_REID_DELTA['hota']:+6.2f}\n"
    f"  ΔMOTA  trackers {r[3] - b[3]:+6.2f}   BoT-SORT paper {BOTSORT_PAPER_REID_DELTA['mota']:+6.2f}   "
    f"gap {(r[3] - b[3]) - BOTSORT_PAPER_REID_DELTA['mota']:+6.2f}\n"
    f"  ΔIDF1  trackers {r[4] - b[4]:+6.2f}   BoT-SORT paper {BOTSORT_PAPER_REID_DELTA['idf1']:+6.2f}   "
    f"gap {(r[4] - b[4]) - BOTSORT_PAPER_REID_DELTA['idf1']:+6.2f}"
)

### 7.2 BoT-SORT — per-sequence vs reference (Table 8 HOTA + Table 13 IDF1)


In [ ]:
REID_STUDY_PER_SEQ = {
    "MOT17-02": {
        "no_reid": {"hota": 47.131, "idf1": 56.968},
        "mot17_th02": {"hota": 49.304, "idf1": 60.0},
    },
    "MOT17-04": {
        "no_reid": {"hota": 78.976, "idf1": 91.021},
        "mot17_th02": {"hota": 79.046, "idf1": 90.864},
    },
    "MOT17-05": {
        "no_reid": {"hota": 60.078, "idf1": 75.124},
        "mot17_th02": {"hota": 61.469, "idf1": 77.969},
    },
    "MOT17-09": {
        "no_reid": {"hota": 67.941, "idf1": 79.985},
        "mot17_th02": {"hota": 65.878, "idf1": 78.832},
    },
    "MOT17-10": {
        "no_reid": {"hota": 57.204, "idf1": 76.157},
        "mot17_th02": {"hota": 59.565, "idf1": 81.087},
    },
    "MOT17-11": {
        "no_reid": {"hota": 66.697, "idf1": 77.326},
        "mot17_th02": {"hota": 66.699, "idf1": 77.326},
    },
    "MOT17-13": {
        "no_reid": {"hota": 69.833, "idf1": 89.533},
        "mot17_th02": {"hota": 69.791, "idf1": 89.431},
    },
}


def ref_seq_key(seq: str) -> str:
    parts = seq.split("-")
    return f"{parts[0]}-{parts[1]}"


for seq in ACTIVE_SEQUENCES:
    key = ref_seq_key(seq)
    ref = REID_STUDY_PER_SEQ.get(key, {})
    print(seq)
    print(f"  {'Config':<28}  {'HOTA':>6}  {'IDF1':>6}  {'IDSW':>5}  {'Ref H':>6}  {'ΔH':>6}  {'Ref I':>6}  {'ΔI':>6}")
    for label, res in botsort_rows:
        hota, assa, idf1, idsw = seq_metrics(res, seq)
        ref_key = "no_reid" if "baseline" in label else "mot17_th02"
        ref_vals = ref.get(ref_key, {})
        ref_hota = ref_vals.get("hota", float("nan"))
        ref_idf1 = ref_vals.get("idf1", float("nan"))
        delta_h = hota - ref_hota if ref_hota == ref_hota else float("nan")
        delta_i = idf1 - ref_idf1 if ref_idf1 == ref_idf1 else float("nan")
        ref_h_s = f"{ref_hota:6.2f}" if ref_hota == ref_hota else "   n/a"
        ref_i_s = f"{ref_idf1:6.2f}" if ref_idf1 == ref_idf1 else "   n/a"
        delta_h_s = f"{delta_h:+6.2f}" if delta_h == delta_h else "   n/a"
        delta_i_s = f"{delta_i:+6.2f}" if delta_i == delta_i else "   n/a"
        print(f"  {label:<28}  {hota:6.2f}  {idf1:6.2f}  {idsw:5d}  {ref_h_s}  {delta_h_s}  {ref_i_s}  {delta_i_s}")
    print()

### 8. Visual comparison — largest ReID gain sequence

Side-by-side **baseline vs +ReID** video for the val sequence with the largest ΔHOTA
(from the runs above). On Colab the mp4 is downloaded automatically.


In [ ]:
# Auto-pick the sequence with the largest HOTA gain (override with COMPARE_SEQ = "MOT17-02-FRCNN").
COMPARE_SEQ: str | None = None
COMPARE_FPS = 30
COMPARE_MAX_FRAMES: int | None = None  # None = full sequence


def _mot_frame_to_detections(mot: dict, frame_idx: int) -> sv.Detections:
    frame = mot.get(frame_idx)
    if frame is None:
        return sv.Detections.empty()
    active = frame.ids >= 0
    if not np.any(active):
        return sv.Detections.empty()
    return sv.Detections(
        xyxy=sv.xywh_to_xyxy(frame.boxes[active]).astype(np.float32),
        tracker_id=frame.ids[active].astype(int),
        confidence=frame.confidences[active].astype(np.float32),
    )


def _annotate_tracks(frame_bgr: np.ndarray, detections: sv.Detections) -> np.ndarray:
    if len(detections) == 0:
        return frame_bgr
    palette, lookup = sv.ColorPalette.DEFAULT, sv.ColorLookup.TRACK
    scene = sv.BoxAnnotator(color=palette, color_lookup=lookup, thickness=2).annotate(frame_bgr, detections)
    labels = [str(int(tid)) for tid in detections.tracker_id]
    return sv.LabelAnnotator(
        color=palette,
        color_lookup=lookup,
        text_color=sv.Color.BLACK,
        text_scale=0.5,
    ).annotate(scene, detections, labels=labels)


def _panel_badge(frame: np.ndarray, text: str, accent: tuple[int, int, int]) -> np.ndarray:
    out = frame.copy()
    font, scale, thick = cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2
    (tw, th), _ = cv2.getTextSize(text, font, scale, thick)
    x, y, pad, bar = 12, 12, 10, 6
    w, h = tw + 2 * pad + bar + 8, th + 2 * pad
    cv2.rectangle(out, (x, y), (x + w, y + h), (24, 24, 28), -1)
    cv2.rectangle(out, (x + 6, y + 6), (x + 6 + bar, y + h - 6), accent, -1)
    cv2.putText(out, text, (x + bar + pad + 4, y + pad + th), font, scale, (245, 245, 245), thick, cv2.LINE_AA)
    return out


seq_gains: list[tuple[str, float, float, float]] = []
for seq in ACTIVE_SEQUENCES:
    h_b, _, i_b, _ = seq_metrics(result_baseline, seq)
    h_r, _, i_r, _ = seq_metrics(result_reid, seq)
    if h_b == h_b and h_r == h_r:
        seq_gains.append((seq, h_r - h_b, i_r - i_b, h_r))

if not seq_gains:
    raise RuntimeError("No per-sequence metrics — run §5–§7 first.")

seq_gains.sort(key=lambda row: row[1], reverse=True)
print("Per-sequence ReID ΔHOTA (largest first):")
for seq, dh, di, _ in seq_gains:
    print(f"  {seq:<20}  ΔHOTA {dh:+6.2f}  ΔIDF1 {di:+6.2f}")

COMPARE_SEQ = COMPARE_SEQ or seq_gains[0][0]
delta_hota, delta_idf1 = next((dh, di) for s, dh, di, _ in seq_gains if s == COMPARE_SEQ)
print(f"\nRendering comparison for {COMPARE_SEQ} (ΔHOTA {delta_hota:+.2f}, ΔIDF1 {delta_idf1:+.2f})")

pred_base = OUTPUT_ROOT / "botsort_baseline" / "preds" / f"{COMPARE_SEQ}.txt"
pred_reid = OUTPUT_ROOT / "botsort_reid" / "preds" / f"{COMPARE_SEQ}.txt"
if not pred_base.is_file() or not pred_reid.is_file():
    raise FileNotFoundError(f"Missing preds for {COMPARE_SEQ}:\n  {pred_base}\n  {pred_reid}")

mot_base = load_mot_file(pred_base)
mot_reid = load_mot_file(pred_reid)
img_dir = SEQUENCE_PATHS[COMPARE_SEQ]["img"]
images = sorted(img_dir.glob("*.jpg"))
n_frames = len(images) if COMPARE_MAX_FRAMES is None else min(len(images), COMPARE_MAX_FRAMES)

sample = cv2.imread(str(images[0]))
if sample is None:
    raise RuntimeError(f"Could not read {images[0]}")
h, w = sample.shape[:2]

out_path = OUTPUT_ROOT / f"compare_{COMPARE_SEQ}_baseline_vs_reid.mp4"
video_info = sv.VideoInfo(width=w * 2, height=h, fps=COMPARE_FPS, total_frames=n_frames)

with sv.VideoSink(str(out_path), video_info) as sink:
    for i in range(n_frames):
        frame_idx = i + 1
        frame = cv2.imread(str(images[i]))
        if frame is None:
            continue
        left = _panel_badge(
            _annotate_tracks(frame.copy(), _mot_frame_to_detections(mot_base, frame_idx)),
            "BASELINE (NO REID)",
            (0, 165, 255),
        )
        right = _panel_badge(
            _annotate_tracks(frame.copy(), _mot_frame_to_detections(mot_reid, frame_idx)),
            "BOT-SORT + REID",
            (80, 200, 120),
        )
        sink.write_frame(np.hstack([left, right]))

print(f"Wrote {out_path} ({n_frames} frames @ {COMPARE_FPS} fps)")
display(Video(str(out_path), embed=True, width=min(960, w)))
if IN_COLAB:
    files.download(str(out_path))